In [3]:
#download example dataset
!wget "http://cseweb.ucsd.edu/~viscomp/projects/LF/papers/ECCV20/nerf/tiny_nerf_data.npz"
#install the requirements
!pip install torch torchvision lightning numpy

--2026-01-29 21:43:58--  http://cseweb.ucsd.edu/~viscomp/projects/LF/papers/ECCV20/nerf/tiny_nerf_data.npz
Resolving cseweb.ucsd.edu (cseweb.ucsd.edu)... 132.239.8.30
Connecting to cseweb.ucsd.edu (cseweb.ucsd.edu)|132.239.8.30|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cseweb.ucsd.edu//~viscomp/projects/LF/papers/ECCV20/nerf/tiny_nerf_data.npz [following]
--2026-01-29 21:43:58--  https://cseweb.ucsd.edu//~viscomp/projects/LF/papers/ECCV20/nerf/tiny_nerf_data.npz
Connecting to cseweb.ucsd.edu (cseweb.ucsd.edu)|132.239.8.30|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12727482 (12M)
Saving to: ‘tiny_nerf_data.npz.1’

tiny_nerf_data.npz. 100%[===================>]  12.14M  6.27MB/s    in 1.9s    

2026-01-29 21:44:00 (6.27 MB/s) - ‘tiny_nerf_data.npz.1’ saved [12727482/12727482]



In [2]:
import torch
import lightning as L

In [3]:
#prepare the dataset
from typing import override
from torch.utils.data import DataLoader, Dataset, random_split
import numpy as np


def get_rays(H, W, focal, c2w):
    """
    Return a set of orgins and directions for camera rays for each pixel.
    
    Args:
        H (int): number of pixels along the verticle axis of the camera.
        W (int): number of pixels along the horizonal axis of the camera.
        focal (float): the focal ratio of the camera.
        c2w (torch.Tensor): the camera to world transformation matrix.
            size: (4, 4, N) #TODO: check these dimensions
    Results:
        torch.tensor: the ray origins.
        
        torch.tensor: the unit direction (magnitude of 1) of the rays.
    """
    i, j = torch.meshgrid(torch.arange(W), torch.arange(H), indexing="xy")
    dirs = torch.stack([(i-W*.5)/focal, -(j-H*.5)/focal, -torch.ones_like(i)], dim=-1)
    
    #these two lines are equivalent to matmul
    rays_d = torch.sum(dirs[..., None, :] * c2w[:3, :3], dim=-1)
    rays_o = torch.broadcast_to(c2w[:3,-1], rays_d.shape)
    return rays_o, rays_d



class TinyDataset(Dataset):
    def __init__(self, file_name):
        super().__init__()

        data = np.load(file_name)
        images = torch.Tensor(data['images'])
        poses = torch.Tensor(data['poses'])
        focal = data['focal']
        H, W = images.shape[1:3]
        rays = [tuple(get_rays(H, W, focal, poses[i])) for i in range(poses.shape[0])]
        self.rays_o = torch.stack([i[0] for i in rays], dim=0)
        self.rays_d = torch.stack([i[1] for i in rays], dim=0)

    def __len__(self):
        return self.rays_o.shape[0]

    def __getitem__(self, index):
        return self.rays_o[index, ...], self.rays_d[index, ...]

tiny_dataset = TinyDataset("tiny_nerf_data.npz")

train_dataset, test_dataset = random_split(tiny_dataset, [96, 10])

batch_size = 100
train_dataloader = DataLoader(train_dataset, batch_size)
test_dataloader = DataLoader(test_dataset, batch_size)

Rendering routine

here, we impliment the sampling and volume rendering techniques detailed in the paper.

Here we impliment so called stratified sampling, draws points from evenly spaces bins along a camera ray. Formaly it is described by equation (3) in the paper.

$$ t_i \sim \text{Uniform} \left[t_n + \frac{i -1}{N}\left(t_f - t_n\right), t_n + \frac{i}{N}\left(t_f - t_n\right)\right]$$

Where $t_i$ is the $i$-th length along a ray. $t_n$ is the $n$-th length of a sample of evenly spaced points between $t_0$ and $t_f$ inclusive. More concretly, this can be thought as the elements of the $n$-th element of 
```python
np.linspace(t_0, t_f, num_points)
```

This implimentation aproaches so called sampling, by first creating a set of evenly spaced points along a ray and then applying noise to them.

So, first we create an array of the left hand side of the bins
```python 
t = torch.linspace(0.0, 1.0, N + 1)[:-1]
```
and then add noise from the standard uniform distrubtion that has been scaled to the bin width.
```python
noi = torch.rand(size=t.shape)
noi = noi.to(near)
t + (bound_diff / N)[..., None] * noi
```

We can think of this mathematically as creating a set
$$
t = \{x (\text{bin width}) | x \in \mathbb{Z} \text{ and } 0 \leq x \leq N\}
$$
and then adding some noice to the elements of this set
$$
\text{noise}_i = u \sim \text{Uniform} \left(0, \text{bin width} \right)
$$
and adding these together
$$
\text{stratified sample}_i = t_i + \text{noise}_i
$$



In [ ]:
def stratified_sampling(N, near, far, noise=False):
    """stratified sampling

    Args:
        N (int): number of a bins 
        near (torch.Tensor): near bound
            size: (batch_size, H, W)
        far (torch.Tensor): far bound. bound_near < bound_far for all elements
            size: (batch_size, H, W)

    Returns:
        torch.Tensor: ray lengths
            size: (batch_size, H, W, N)
    """
    batch_size = near.shape[0]
    t = torch.linspace(0.0, 1.0, N + 1)[:-1]
    t = t.to(near)
    #t = torch.stack([t for _ in range(batch_size)], dim=0)
    t = torch.broadcast_to(t[None, None, None, :] , size=(*near.shape, N))
    bound_diff = far - near
    t = bound_diff[..., None] * t + near[..., None]
    noi = torch.rand(size=t.shape)
    noi = noi.to(near)
    if noise:
        return t + (bound_diff / N)[..., None] * noi
    else:
        return t


volume render

here we take the volume rending equation, the paper's equation (1)
$$
\vec{C}(\vec{r}) = \int_{t_n}^{t_f} T(t) \sigma(\vec{r}(t)) \vec{c}(\vec{r}(t), \vec{d}) \text{d}t, \text{ where } T(t) = \exp \left(- \int_{t_n}^{t_f} \sigma(\vec{r}(s)) \text{d}s \right)
$$
and evaluate numerically with

$$
\vec{\hat{C}}(\vec{r}) = \sum_{i=0}^{N} T_i \left(1-\exp(-\sigma_i \delta_i)\right) \vec{c}_i, \text{ where } T_i = \exp \left(-\sum_{j=1}^{i-1}\sigma_j \delta_i\right)
$$

In [ ]:

def render(model, rays_o, rays_d, bounds, N_c=64 ):
    """

    Args:
        model (torch.nn.Module): the model to query points for color and density
        rays_o (torch.Tensor): ray origins
            size: (batch_size, H, W, 3)
        rays_d (torch.Tensor): ray directions
            size: (batch_size, H, W, 3)
        bounds: [near, far] each a tensor of shape [batch_size, H, W]
        N_c (int): the number of points to query along each ray

    Returns:
        torch.Tensor: 
            size:  (batch_size, H, W)
    """
    near = bounds[0]
    far = bounds[1]
    t = stratified_sampling(N_c, near, far, noise=True) #[batch_size, H, W, N_c]
    d = torch.broadcast_to(rays_d[..., None], size=(*rays_d.shape, N_c))
    #points
    x = rays_d[..., None, :] * t[..., None] + rays_o[..., None, :]
    
    colors, densities = model(x, d)

    C_hat, w_i, alpha_i = volume_render(t, colors, densities)

    return C_hat